#### 1) Notebook Setup

In [1]:
from pathlib import Path
import sys
import os

# Locate project root (the folder containing requirements.txt)
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "requirements.txt").exists():
    raise FileNotFoundError(
        "Could not find project root. Check the notebook's working directory."
    )

# Make project scripts importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

Project root: c:\Users\adminim\Desktop\waterlogging_ml
Python: c:\Users\adminim\Desktop\waterlogging_ml\.venv\Scripts\python.exe


#### 2) Configuring the three study areas (SilkBoard , Bellandur & Nayandahalli)

In [2]:
AREAS = {
    "silk_board": {
        "latitude": 12.917,
        "longitude": 77.623,
        "radius": 2000
    },

    "bellandur": {
        "latitude": 12.926088,
        "longitude": 77.675778,
        "radius": 2000
    },

    "nayandahalli": {
        "latitude": 12.93961,
        "longitude": 77.52601,
        "radius": 2000
    }
}

# Select the area to collect data for
AREA = "bellandur"

# Load selected configuration
AREA_CONFIG = AREAS[AREA]

LATITUDE = AREA_CONFIG["latitude"]
LONGITUDE = AREA_CONFIG["longitude"]
RADIUS = AREA_CONFIG["radius"]

# Area-specific output directories
RAW_DIR = PROJECT_ROOT / "data" / AREA / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / AREA / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Selected area:", AREA)
print("Coordinates:", LATITUDE, LONGITUDE)
print("Radius:", RADIUS, "metres")
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)

Selected area: bellandur
Coordinates: 12.926088 77.675778
Radius: 2000 metres
Raw directory: c:\Users\adminim\Desktop\waterlogging_ml\data\bellandur\raw
Processed directory: c:\Users\adminim\Desktop\waterlogging_ml\data\bellandur\processed


#### Verifying all the .py scripts before building the pipeline

In [4]:
scripts_to_inspect = [
    "download_roads.py",
    "download_terrain.py",
    "download_landcover.py",
    "download_rainfall.py",
    "extract_rainfall_events.py",
    "process_bbmp_kml.py",
    "build_combined_bbmp_gpkg.py",
    "extract_road_features.py",
    "match_bbmp_to_roads.py",
    "build_bbmp_road_features.py"
]

for filename in scripts_to_inspect:
    path = PROJECT_ROOT / filename

    print(f"\n{'='*65}")
    print(f"FILE: {filename}")
    print('='*65)

    if path.exists():
        lines = path.read_text(encoding="utf-8").splitlines()
        print("\n".join(lines[:35]))
    else:
        print("File not found")


FILE: download_roads.py
import osmnx as ox
from pathlib import Path

# Study area
CENTER = (12.917, 77.623)
DISTANCE_METERS = 2000

output_dir = Path("data/raw/roads")
output_dir.mkdir(parents=True, exist_ok=True)

print("Downloading road network around Silk Board...")

# Download drivable roads
G = ox.graph_from_point(
    CENTER,
    dist=DISTANCE_METERS,
    network_type="drive",
    simplify=True,
    retain_all=True
)

# Convert network edges into individual road segments
nodes, edges = ox.graph_to_gdfs(
    G,
    nodes=True,
    edges=True
)

# Give each road segment a stable ID within this download
edges = edges.reset_index()
edges["road_id"] = (
    edges["u"].astype(str) + "_" +
    edges["v"].astype(str) + "_" +
    edges["key"].astype(str)
)

FILE: download_terrain.py
import ee

# 1. Initialize Earth Engine
PROJECT_ID = "waterloggingbengaluru"
ee.Initialize(project=PROJECT_ID)

# 2. Define Silk Board location
silk_board = ee.Geometry.Point([77.623, 12.917])

# Approximatel